# =========================
# IMPORTS LIBRAIRIES
# =========================

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# =========================
# LOAD DATASET FEATURE ENGINEERED
# =========================

In [2]:
df = pd.read_parquet("../data/featured_data.parquet")

# Normaliser les noms de colonnes (sécurité)
df.columns = df.columns.str.lower()

# Vérification rapide
print("Shape du dataset :", df.shape)
df.head()

Shape du dataset : (339013, 29)


,month,day_of_month,scheduled_departure_time,scheduled_arrival_time,unique_carrier,flight_number,scheduled_elapsed_time,departure_delay,origin,dest,...,date,year,day,day_of_week_num,is_weekend,departure_hour,is_peak_hour,airport_traffic,carrier_traffic,short_flight
0,1,4,1125,1240,WN,746,55.0,78.0,ABQ,AMA,...,2008-01-04,2008,4,4,0,11,0,1967,58324,1
1,1,4,545,880,WN,2126,215.0,1.0,ABQ,BWI,...,2008-01-04,2008,4,4,0,5,0,1967,58324,0
2,1,4,875,1030,WN,45,95.0,9.0,ABQ,DAL,...,2008-01-04,2008,4,4,0,8,1,1967,58324,1
3,1,4,415,570,WN,87,95.0,1.0,ABQ,DAL,...,2008-01-04,2008,4,4,0,4,0,1967,58324,1
4,1,4,1215,1360,WN,230,85.0,108.0,ABQ,DAL,...,2008-01-04,2008,4,4,0,12,0,1967,58324,1


# =========================
# GESTION DES VALEURS MANQUANTES
# =========================

# Ces colonnes représentent des types de retard
# NaN signifie "aucun retard", donc on remplace par 0

In [3]:
delay_cols = [
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay"
]

df[delay_cols] = df[delay_cols].fillna(0)

# Supprimer les autres valeurs manquantes éventuelles
df = df.dropna()

print("Après nettoyage :", df.shape)

Après nettoyage : (338972, 29)


# =========================
# ENCODAGE DES VARIABLES CATEGORIELLES
# =========================

In [4]:
cat_cols = ["origin", "dest", "unique_carrier"]

encoders = {}  # pour sauvegarder les encodeurs

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

print("Encodage terminé.")

Encodage terminé.


# =========================
# SEPARATION FEATURES / TARGET
# =========================


In [5]:
# Target = variable à prédire
y = df["taxi_out"]

# Features = toutes les colonnes sauf target + date
X = df.drop(columns=["taxi_out", "date"])

print("X shape :", X.shape)
print("y shape :", y.shape)

X shape : (338972, 27)
y shape : (338972,)


# =========================
# SPLIT DATASET
# =========================

In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

Train : (237280, 27)
Validation : (50846, 27)
Test : (50846, 27)


# =========================
# STANDARDISATION DES DONNEES
# =========================


In [7]:
scaler = StandardScaler()

# Fit uniquement sur train (IMPORTANT pour éviter data leakage)
X_train = scaler.fit_transform(X_train)

# Transformation validation et test
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Scaling terminé.")

Scaling terminé.


# =========================================================
# SAVE PROCESSED DATA FOR DL MODELS
# =========================================================

In [8]:
np.save("../data/X_train.npy", X_train)
np.save("../data/X_val.npy", X_val)
np.save("../data/X_test.npy", X_test)

np.save("../data/y_train.npy", y_train)
np.save("../data/y_val.npy", y_val)
np.save("../data/y_test.npy", y_test)

print("Datasets DL sauvegardés ✔")

Datasets DL sauvegardés ✔


# =========================
# VERIFICATION FINALE
# =========================

In [9]:
print("FINAL SHAPES")
print("---------------------")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDataset prêt pour Deep Learning ✔")

FINAL SHAPES
---------------------
Train: (237280, 27)
Validation: (50846, 27)
Test: (50846, 27)

Dataset prêt pour Deep Learning ✔
